# 06 · Calibration

## What this notebook covers
A model that outputs P(y=1|x) = 0.8 should be correct ~80% of the time. Calibration measures how well predicted probabilities match empirical frequencies. Miscalibration means your confidence estimates are wrong — a critical failure for risk scoring, decision thresholds, and any downstream system that uses raw probabilities.

## Why calibration matters
Gradient-boosted trees are typically *overconfident* (probabilities cluster near 0 and 1). Neural networks trained with cross-entropy can be well-calibrated early but become overconfident as training continues. Calibration is often an afterthought but it is essential for:

- **Decision thresholds**: a threshold chosen at 0.5 on miscalibrated probabilities may be optimal at a completely different operating point
- **Cost-sensitive learning**: expected cost calculations require accurate probabilities
- **RAG / LLM uncertainty**: when plugging model confidence into retrieval or chain-of-thought systems

## Calibration techniques
| Method | Notes |
|--------|-------|
| Temperature scaling | Single parameter; post-hoc; requires calibration set |
| Platt scaling | Logistic regression on logits; effective for SVMs / shallow models |
| Isotonic regression | Non-parametric; can overfit on small calibration sets |

Temperature scaling (TS) is the modern default for neural networks and gradient boosted models. It minimises NLL on a held-out calibration set with a single scalar T, rescaling logits: p_calibrated = softmax(logits / T).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibrationDisplay, calibration_curve
from sklearn.metrics import log_loss, brier_score_loss
from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

X, y = make_classification(n_samples=3000, n_features=20, n_informative=12, random_state=0)
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, random_state=0)
X_cal, X_te, y_cal, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=0)

clf = GradientBoostingClassifier(n_estimators=200, random_state=0)
clf.fit(X_tr, y_tr)
proba_raw = clf.predict_proba(X_te)[:, 1]
proba_cal_raw = clf.predict_proba(X_cal)[:, 1]

def ece(y_true, y_prob, n_bins=10):
    """Expected Calibration Error (equal-width bins)."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece_val = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        bin_acc  = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()
        ece_val += mask.sum() * abs(bin_acc - bin_conf)
    return ece_val / len(y_true)

print(f"ECE (raw)   : {ece(y_te, proba_raw):.4f}")
print(f"Brier score : {brier_score_loss(y_te, proba_raw):.4f}")
print(f"Log-loss    : {log_loss(y_te, proba_raw):.4f}")

In [ ]:
def fit_temperature(y_cal, p_cal):
    """Find optimal temperature T by minimising NLL on calibration set."""
    logits_cal = logit(np.clip(p_cal, 1e-7, 1 - 1e-7))

    def nll(T):
        scaled = expit(logits_cal / T)
        return log_loss(y_cal, scaled)

    result = minimize_scalar(nll, bounds=(0.1, 10.0), method="bounded")
    return result.x

T = fit_temperature(y_cal, proba_cal_raw)
print(f"Optimal temperature T = {T:.4f}")

logits_te = logit(np.clip(proba_raw, 1e-7, 1 - 1e-7))
proba_temp = expit(logits_te / T)
print(f"ECE after temperature scaling: {ece(y_te, proba_temp):.4f}")

In [ ]:
# Platt scaling
platt = LogisticRegression()
platt.fit(logit(np.clip(proba_cal_raw, 1e-7, 1 - 1e-7)).reshape(-1, 1), y_cal)
proba_platt = platt.predict_proba(logit(np.clip(proba_raw, 1e-7, 1 - 1e-7)).reshape(-1, 1))[:, 1]
print(f"ECE after Platt scaling: {ece(y_te, proba_platt):.4f}")

# Reliability diagram comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (label, p) in zip(axes, [("Raw GBM", proba_raw), ("Temp scaling", proba_temp), ("Platt scaling", proba_platt)]):
    CalibrationDisplay.from_predictions(y_te, p, n_bins=10, ax=ax, name=label)
    ax.set_title(f"{label}  (ECE={ece(y_te, p):.3f})")
plt.tight_layout()
plt.savefig(f"{base}/06_calibration.png", dpi=120)
plt.show()

In [ ]:
class HonestAbe:
    """Production calibration monitor. Alerts when ECE drifts above threshold."""
    def __init__(self, ece_threshold=0.05, n_bins=10):
        self.ece_threshold = ece_threshold
        self.n_bins = n_bins
        self.history = []

    def check(self, y_true, y_prob, window_name="batch"):
        """Check calibration and record result."""
        current_ece = ece(np.array(y_true), np.array(y_prob), self.n_bins)
        alert = current_ece > self.ece_threshold
        entry = {"window": window_name, "ece": round(current_ece, 4), "alert": alert}
        self.history.append(entry)
        if alert:
            print(f"[ALERT] {window_name}: ECE={current_ece:.4f} exceeds threshold {self.ece_threshold}")
        else:
            print(f"[OK]    {window_name}: ECE={current_ece:.4f}")
        return entry

    def report(self):
        return pd.DataFrame(self.history)

abe = HonestAbe(ece_threshold=0.04)
abe.check(y_te, proba_raw,  window_name="raw_model")
abe.check(y_te, proba_temp, window_name="temp_scaled")
abe.check(y_te, proba_platt, window_name="platt_scaled")
print(abe.report().to_string(index=False))